# 🎬 דיבוב לעברית בקולות המקוריים – אווטאר: שבעת המקלטים (עונה 1 פרק 1)
מחברת אוטומטית מלאה להרצת מודל **`Yzamari/f5tts-hebrew-v2`** (שכפול קול Zero-Shot בעברית עם ניקוד פונטי של `Phonikud`).

### 🚀 איך להריץ בלחיצה אחת:
1. ודאו שמוגדר GPU: בתפריט העליון לחצו על **Runtime -> Change runtime type** ובחרו **T4 GPU**.
2. לחצו על **Runtime -> Run all** (או `Ctrl+F9` / `Cmd+F9`)! אין צורך להעלות שום קובץ ידנית – המחברת מושכת אוטומטית את כל דגימות הקול והתסריט!

In [ ]:
# @title 1. התקנת ספריות וטעינת מודל F5-TTS Hebrew v2
!pip install -q f5-tts safetensors huggingface_hub soundfile torchaudio

import os
import json
import torch
import torchaudio
import soundfile as sf
import numpy as np
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from f5_tts.model import CFM, DiT
from f5_tts.model.utils import get_tokenizer
from f5_tts.infer.utils_infer import load_vocoder, target_rms, hop_length

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on device: {device}')

print('Downloading Yzamari/f5tts-hebrew-v2 weights and vocab.txt...')
ckpt_path = hf_hub_download(repo_id='Yzamari/f5tts-hebrew-v2', filename='model.safetensors')
vocab_path = hf_hub_download(repo_id='Yzamari/f5tts-hebrew-v2', filename='vocab.txt')

vocab_char_map, vocab_size = get_tokenizer(vocab_path, 'custom')
print(f'Loaded Hebrew vocab size: {vocab_size}')

model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
mel_spec_kwargs = dict(
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    n_mel_channels=100,
    target_sample_rate=24000,
    mel_spec_type='vocos'
)

model = CFM(
    transformer=DiT(**model_cfg, text_num_embeds=vocab_size, mel_dim=100),
    mel_spec_kwargs=mel_spec_kwargs,
    odeint_kwargs=dict(method='euler'),
    vocab_char_map=vocab_char_map,
).to(device)

state_dict = load_file(ckpt_path, device=device)
model.load_state_dict(state_dict, strict=True)
model.eval()

vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=device)
print('✅ Model and Vocos vocoder loaded successfully!')

In [ ]:
# @title 2. הורדה אוטומטית של דגימות הקול והתסריט (ללא צורך בהעלאה ידנית!)
import zipfile
import urllib.request
from google.colab import files

GITHUB_ZIP_URL = 'https://raw.githubusercontent.com/barelgrabli/avatar-hebrew-dubbing/main/dibob_colab_inputs.zip'

if not os.path.exists('dibob_colab_inputs.zip'):
    print('⬇️ מוריד אוטומטית את dibob_colab_inputs.zip מ-GitHub...')
    try:
        urllib.request.urlretrieve(GITHUB_ZIP_URL, 'dibob_colab_inputs.zip')
        print('✅ ירד בהצלחה מ-GitHub!')
    except Exception as e:
        print('לא נמצא ב-GitHub, נא להעלות ידנית:', e)
        uploaded = files.upload()

with zipfile.ZipFile('dibob_colab_inputs.zip', 'r') as zf:
    zf.extractall('.')

with open('subtitles/Avatar_Seven_Havens_S01E01_dubbing_script.json', 'r', encoding='utf-8') as f:
    dubbing_script = json.load(f)

with open('voice_refs/speakers_manifest.json', 'r', encoding='utf-8') as f:
    speakers_manifest = json.load(f)

print(f'✅ נטענו {len(dubbing_script)} שורות דיבוב ו-{len(speakers_manifest)} דגימות קול של דמויות!')

In [ ]:
# @title 3. פונקציית סינתזה + בדיקת שמיעה לדמויות הראשיות (פאבי, ג'יי, קארתיק)
from IPython.display import Audio, display

@torch.inference_mode()
def synthesize_hebrew_line(ref_wav_path, ref_text, gen_text_nikud, nfe_step=32, cfg_strength=2.0, sway_sampling_coef=-1.0, speed=1.0):
    audio, sr = torchaudio.load(ref_wav_path)
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    if sr != 24000:
        audio = torchaudio.transforms.Resample(sr, 24000)(audio)

    max_samples = int(8.0 * 24000)
    if audio.shape[-1] > max_samples:
        audio = audio[:, :max_samples]

    rms = torch.sqrt(torch.mean(torch.square(audio)))
    if rms < target_rms:
        audio = audio * target_rms / rms

    audio = audio.to(device)
    full_text = ref_text.strip() + ' ' + gen_text_nikud.strip()
    text_list = [list(full_text)]

    ref_audio_len = audio.shape[-1] // hop_length
    ref_char_len = max(len(ref_text.strip()), 10)
    gen_char_len = len(gen_text_nikud.strip())
    estimated_gen_frames = int(max(ref_audio_len * (gen_char_len / ref_char_len) / speed, 24000 * 1.1 // hop_length))
    duration = ref_audio_len + estimated_gen_frames

    generated, _ = model.sample(
        cond=audio,
        text=text_list,
        duration=duration,
        steps=nfe_step,
        cfg_strength=cfg_strength,
        sway_sampling_coef=sway_sampling_coef,
    )

    generated = generated.to(torch.float32)
    generated = generated[:, ref_audio_len:, :]
    generated_mel_spec = generated.permute(0, 2, 1)
    generated_wave = vocoder.decode(generated_mel_spec)
    if rms < target_rms:
        generated_wave = generated_wave * rms / target_rms
    return generated_wave.squeeze().cpu().numpy()

test_dub_ids = [8, 22, 62]  # פאבי, ג'יי, קארתיק
for item in dubbing_script:
    if item['dub_id'] in test_dub_ids:
        spk = item['speaker']
        meta = speakers_manifest[spk]
        ref_text_short = meta['ref_text_en'][:110]
        print(f"🎙️ [{spk}] {item['text_he']}  ({item['text_he_nikud']})")
        wav_out = synthesize_hebrew_line(meta['ref_audio_path'], ref_text_short, item['text_he_nikud'])
        display(Audio(wav_out, rate=24000))

In [ ]:
# @title 4. יצירת כל 290 שורות הדיבוב של הפרק והורדה אוטומטית
from tqdm.auto import tqdm

os.makedirs('dubbed_lines', exist_ok=True)

for item in tqdm(dubbing_script, desc='Generating Hebrew Dub'):
    dub_id = item['dub_id']
    out_path = f'dubbed_lines/dub_{dub_id:03d}.wav'
    if os.path.exists(out_path):
        continue

    spk = item['speaker']
    meta = speakers_manifest[spk]
    if meta['duration_sec'] < 3.0:
        fallback_spk = 'Pavi' if item['gender'] == 'F' else 'Jae'
        meta = speakers_manifest[fallback_spk]

    ref_text_short = meta['ref_text_en'][:110]
    try:
        wav_out = synthesize_hebrew_line(
            ref_wav_path=meta['ref_audio_path'],
            ref_text=ref_text_short,
            gen_text_nikud=item['text_he_nikud'],
            nfe_step=32,
            cfg_strength=2.0
        )
        sf.write(out_path, wav_out, 24000)
    except Exception as e:
        print(f'Warning on line {dub_id}: {e}')

print('📦 אורז את כל קובצי הדיבוב ל-dubbed_lines.zip...')
!zip -r -q dubbed_lines.zip dubbed_lines/
print('✅ מוכן! מוריד את dubbed_lines.zip למחשב שלך...')
files.download('dubbed_lines.zip')